In [ ]:
%pip install dragoneye-python numpy Pillow matplotlib

In [ ]:
import os
from dragoneye import NormalizedBbox

from PIL import Image, ImageDraw, ImageFont
import zlib
from dragoneye import Dragoneye
from dragoneye.models import (
    ClassificationPredictImageResponse,
)
from dragoneye.types.media import Image as DragoneyeImage
from pathlib import Path
import matplotlib.pyplot as plt

from dragoneye.models import ClassificationObjectPrediction
from IPython.display import display as IPythonDisplay  # pyright: ignore [reportUnknownVariableType]

In [ ]:
# Configuration - Update these paths for your use case
LOCAL_IMAGE_PATH = "/path/to/image.jpg"

# Get API key from environment variable for security
API_KEY = os.environ.get("DRAGONEYE_API_KEY")
if not API_KEY:
    raise ValueError(
        "Please set DRAGONEYE_API_KEY environment variable. "
        "For this notebook, you can use: os.environ['DRAGONEYE_API_KEY'] = 'your-key-here'"
    )

MODEL_NAME = "recognize_anything/model_name"


# Initialize client and load media
client = Dragoneye(api_key=API_KEY)
media = DragoneyeImage.from_path(LOCAL_IMAGE_PATH)

# Load or fetch predictions
prediction_result = await client.classification.predict_image(
    media=media,
    model_name=MODEL_NAME,
)

In [ ]:
# Display and styling constants

LABEL_FONT_SIZE = 16
LABEL_PADDING = 4
LABEL_ALPHA = 128  # 0-255 for PIL (0 = transparent, 255 = opaque)
BBOX_THICKNESS_RATIO = 0.003  # Ratio of frame dimension to bbox thickness
MIN_BBOX_THICKNESS = 2
SHADOW_OFFSET = 2


def generate_deterministic_color(name: str) -> tuple[int, int, int]:
    """
    Generate a deterministic RGB color from a category name using hashing.

    Uses CRC32 to ensure consistent colors across runs for the same category name.
    Colors are generated to avoid being too dark (minimum brightness per channel).

    Args:
        name: Category name to generate color for

    Returns:
        Tuple of (red, green, blue) values in range [80, 207] for PIL RGB format

    Example:
        >>> generate_deterministic_color("trash")
        (201, 165, 142)
    """
    COLOR_CHANNEL_MASK = 0x7F
    COLOR_CHANNEL_OFFSET = 80

    hash_value = zlib.crc32(name.encode())

    # Spread hash across RGB channels and ensure minimum brightness
    red = ((hash_value >> 14) & COLOR_CHANNEL_MASK) + COLOR_CHANNEL_OFFSET
    green = ((hash_value >> 7) & COLOR_CHANNEL_MASK) + COLOR_CHANNEL_OFFSET
    blue = ((hash_value >> 0) & COLOR_CHANNEL_MASK) + COLOR_CHANNEL_OFFSET

    return (red, green, blue)


def denormalize_bbox(
    normalized_bbox: NormalizedBbox,
    image_width: int,
    image_height: int,
) -> tuple[int, int, int, int]:
    """
    Convert normalized bbox coordinates [0, 1] to absolute pixel coordinates.

    Args:
        normalized_bbox: Bounding box with coordinates in [0, 1] range as [x1, y1, x2, y2]
        image_width: Width of the image in pixels
        image_height: Height of the image in pixels

    Returns:
        Tuple of (x1, y1, x2, y2) in absolute pixel coordinates, clamped to image bounds

    Example:
        >>> denormalize_bbox([0.1, 0.2, 0.9, 0.8], 1920, 1080)
        (192, 216, 1728, 864)
    """
    x1 = int(round(normalized_bbox[0] * image_width))
    y1 = int(round(normalized_bbox[1] * image_height))
    x2 = int(round(normalized_bbox[2] * image_width))
    y2 = int(round(normalized_bbox[3] * image_height))

    # Clamp to image boundaries
    x1 = max(0, min(image_width - 1, x1))
    y1 = max(0, min(image_height - 1, y1))
    x2 = max(0, min(image_width - 1, x2))
    y2 = max(0, min(image_height - 1, y2))

    return x1, y1, x2, y2


def format_prediction_label(prediction: ClassificationObjectPrediction) -> str:
    """
    Format a prediction into a display label string.

    Args:
        prediction: Classification prediction object

    Returns:
        Formatted label string with category name and optional score

    Example:
        >>> format_prediction_label(prediction)
        "trash_bag 0.95"
    """
    category = prediction.category
    name = category.displayName or category.name or "object"
    score = category.score

    if isinstance(score, (int, float)):
        return f"{name} {score:.2f}"
    return name


def draw_label_on_image(
    draw: ImageDraw.ImageDraw,
    x: int,
    y: int,
    label: str,
    color: tuple[int, int, int],
    image_width: int,
    image_height: int,
    font: ImageFont.FreeTypeFont | ImageFont.ImageFont | None = None,
) -> None:
    """
    Draw a filled label box with text on an image using PIL.

    Label is positioned above the bounding box by default, or inside if too close to top edge.
    This function modifies the draw object in place.

    Args:
        draw: PIL ImageDraw object to draw on
        x: X coordinate of the label anchor point (top-left of bbox)
        y: Y coordinate of the label anchor point (top-left of bbox)
        label: Text to display
        color: RGB color tuple for the label background
        image_width: Width of the image
        image_height: Height of the image
        font: PIL Font object (optional, uses default if None)

    Example:
        >>> draw_label_on_image(draw, 100, 100, "trash", (255, 0, 0), 1920, 1080)
    """
    # Get text bounding box
    bbox = draw.textbbox((0, 0), label, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    # Determine label position (above bbox or inside if not enough space)
    label_y_top = y - text_height - LABEL_PADDING * 2
    if label_y_top < 0:
        label_y_top = y + LABEL_PADDING

    label_x_end = min(image_width - 1, x + text_width + LABEL_PADDING * 2)
    label_y_end = min(image_height - 1, label_y_top + text_height + LABEL_PADDING * 2)

    # Draw semi-transparent filled rectangle for label background
    # Create a semi-transparent color by adding alpha channel
    color_with_alpha = color + (LABEL_ALPHA,)
    draw.rectangle(
        [(x, label_y_top), (label_x_end, label_y_end)],
        fill=color_with_alpha,
    )

    # Draw text on top of background
    text_y = label_y_top + LABEL_PADDING
    draw.text(
        (x + LABEL_PADDING, text_y),
        label,
        fill=(0, 0, 0),  # Black text for contrast
        font=font,
    )


def draw_prediction_on_image(
    image: Image.Image,
    prediction: ClassificationObjectPrediction,
) -> None:
    """
    Draw a single prediction (bbox + label) on a PIL Image.

    This function modifies the image in place.

    Args:
        image: PIL Image object
        prediction: Classification prediction to visualize
    """
    image_width, image_height = image.size

    # Create a drawing context with alpha support
    if image.mode != "RGBA":
        image_rgba = image.convert("RGBA")
        overlay = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(overlay)
    else:
        image_rgba = image
        overlay = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(overlay)

    # Extract prediction information
    x1, y1, x2, y2 = denormalize_bbox(
        prediction.normalizedBbox, image_width, image_height
    )
    label = format_prediction_label(prediction)
    category_name = (
        prediction.category.displayName or prediction.category.name or "object"
    )
    color = generate_deterministic_color(category_name)

    # Calculate adaptive thickness based on image size
    thickness = max(
        MIN_BBOX_THICKNESS,
        int(round(BBOX_THICKNESS_RATIO * max(image_width, image_height))),
    )

    # Try to load a font, fall back to default if unavailable
    font = ImageFont.load_default(size=LABEL_FONT_SIZE)

    # Draw bounding box with shadow for depth on the overlay
    # Shadow
    for i in range(thickness + SHADOW_OFFSET):
        draw.rectangle(
            [(x1 - i, y1 - i), (x2 + i, y2 + i)],
            outline=(0, 0, 0, 255),
            width=1,
        )

    # Main box
    for i in range(thickness):
        draw.rectangle(
            [(x1 - i, y1 - i), (x2 + i, y2 + i)],
            outline=color + (255,),
            width=1,
        )

    # Draw label
    draw_label_on_image(draw, x1, y1, label, color, image_width, image_height, font)

    # Composite the overlay onto the original image
    if image.mode != "RGBA":
        image_rgba = image.convert("RGBA")

    image_rgba = Image.alpha_composite(image_rgba, overlay)

    # Convert back to original mode if necessary
    if image.mode != "RGBA":
        result = image_rgba.convert(image.mode)
        image.paste(result)
    else:
        image.paste(image_rgba)

In [ ]:
def visualize_image_predictions(
    image_path: str,
    predictions_response: ClassificationPredictImageResponse,
    display: bool = True,
) -> Image.Image:
    """
    Visualize predictions on an image using PIL.

    Args:
        image_path: Path to the input image
        predictions_response: Predictions to visualize
        output_path: Optional path to save the output image
        display: Whether to display the image using matplotlib

    Returns:
        PIL Image with predictions drawn

    Raises:
        FileNotFoundError: If input image doesn't exist
    """
    if not Path(image_path).exists():
        raise FileNotFoundError(f"Could not find image at {image_path}")

    # Load image using PIL
    image = Image.open(image_path)

    # Convert to RGBA for transparent overlays
    if image.mode != "RGBA":
        image = image.convert("RGBA")

    # Draw all predictions (modifies image in place)
    for prediction in predictions_response.predictions:
        draw_prediction_on_image(image, prediction)

    # Convert back to RGB for saving/display
    if image.mode == "RGBA":
        # Create a white background
        background = Image.new("RGB", image.size, (255, 255, 255))
        background.paste(image, mask=image.split()[3])  # Use alpha channel as mask
        image = background

    # Display if requested
    if display:
        IPythonDisplay(image)

    return image


visualized_image = visualize_image_predictions(
    LOCAL_IMAGE_PATH,
    prediction_result,
    display=True,
)